# Ch2 심화 — 도구를 LLM이 실제로 보는 법 (@tool · tool_choice · structured output)

본문에서 `create_agent`와 `StateGraph`로 흐름을 짰습니다. 이 노트북은 그 아래 한 겹,
**개발자가 실무에서 매번 부딪히는 세 제어점**를 셀 단위로 직접 돌려 봅니다.

1. **`@tool`의 두 얼굴** — 함수가 LLM에게 넘어갈 때 스키마는 *어디서* 만들어지나. `docstring`+타입힌트 추론 vs `args_schema` 선언, 무엇이 어떻게 달라지나.
2. **`tool_choice`** — 도구를 부를지 말지 *누가* 정하나. `create_agent`엔 이 제어가 없다 — 그럼 강제는 어떻게 하나.
3. **structured output** — 모델 출력을 타입으로 받는 법. 우리 `classify_one`의 수동 JSON 파싱을 한 줄로 지운다.

> 실행: VSCode에서 이 파일을 열고 커널을 `.venv`로 맞춘 뒤 위에서부터 실행합니다.
> 실험 1은 키 없이 됩니다. 실험 2·3은 레포 루트 `.env`의 `OPENROUTER_API_KEY`가 필요합니다.
> 각 실험 끝의 **✏️ 직접 해보기**는 손으로 바꿔 다시 돌려 보라는 뜻입니다 — 눈으로만 읽으면 남지 않습니다.

In [ ]:
# 셋업 — 레포 루트를 찾고 .env를 읽는다(노트북 cwd가 어디든 동작)
import os, sys, json
from pathlib import Path
from dotenv import dotenv_values

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
for p in (root, root / "ch1-llm-basics"):
    sys.path.insert(0, str(p))
# load_dotenv는 이미 설정된 셸 환경을 덮지 않는다 → dotenv_values로 파일 값을 직접 주입
for k, v in dotenv_values(root / ".env").items():
    if v:
        os.environ[k] = v

KEY = os.environ.get("OPENROUTER_API_KEY")
HAS_KEY = bool(KEY) and KEY != "sk-or-..."
BASE = "https://openrouter.ai/api/v1"
MODEL = os.environ.get("ANALYST_MODEL", "google/gemini-3.1-flash-lite")
print("repo root:", root.name, "| 키:", "있음" if HAS_KEY else "없음(실험 2·3은 키 필요)", "| 모델:", MODEL)

## 실험 1 — `@tool`의 두 얼굴: 스키마는 어디서 오나

`@tool`을 붙이면 파이썬 함수가 `{name, description, parameters}` JSON Schema로 바뀌어 API의 `tools` 필드에 실립니다.
모델은 그 스키마만 보고 "언제 · 어떤 인자로" 부를지 정합니다. **스키마가 곧 모델이 아는 전부**입니다.

그 스키마를 만드는 길은 둘입니다. 먼저 **추론 방식** — 함수 시그니처와 docstring에서 자동으로 뽑아냅니다.

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# ① 추론 방식: 타입힌트 → parameters, docstring → description
@tool
def get_weather(city: str, days: int = 3) -> str:
    """도시의 향후 며칠 날씨를 조회한다."""
    return "..."

print("name        :", get_weather.name)
print("description :", get_weather.description)      # ← docstring이 그대로 온다
print("parameters  :", json.dumps(get_weather.args_schema.model_json_schema()["properties"],
                                   ensure_ascii=False))
# days에는 '무슨 뜻인지' 설명이 없다 — 타입과 기본값뿐. 모델은 days가 뭘 의미하는지 모른다.

이제 **선언 방식** — `args_schema`에 Pydantic 모델을 명시합니다. 인자마다 `Field(description=...)`로
설명을 달 수 있어, 모델이 각 인자의 *의미*까지 봅니다. docstring은 여전히 도구 전체의 description으로 남습니다.

In [ ]:
# ② 선언 방식: args_schema=Pydantic → 인자마다 description을 실어 보낸다
class WeatherInput(BaseModel):
    city: str = Field(description="조회할 도시명. 예: 'Seoul'")
    days: int = Field(default=3, description="오늘부터 예보할 일수(1~7)")

@tool(args_schema=WeatherInput)
def get_weather_declared(city: str, days: int = 3) -> str:
    """도시의 향후 며칠 날씨를 조회한다."""
    return "..."

props = get_weather_declared.args_schema.model_json_schema()["properties"]
print("parameters  :", json.dumps(props, ensure_ascii=False, indent=2))
# 이제 city·days에 'description'이 붙는다 → 모델이 인자의 의미까지 읽는다.

**관찰.** 두 도구는 같은 함수지만, 모델에 전달되는 스키마가 다릅니다.
- **추론 방식**: 빠르고 간결. 인자 의미는 이름·타입에만 의존.
- **선언 방식**: `Field(description=...)`로 인자 하나하나에 설명을 실어 보냄 → 모델이 헷갈리는 인자를 정확히 채우게 만드는 실무 무기. 우리 코드도 `check_receipt_sum`을 `@tool(args_schema=CheckReceiptSumInput)`로 선언합니다(`ch1-llm-basics/classify_one.py`).

> **✏️ 직접 해보기.** `WeatherInput`의 `days` 설명을 `"예보 일수. 최대 14"`로 바꾸고 다시 실행해, 스키마 JSON의 `description`이 바뀌는지 확인하세요. 이 문자열이 **매 API 호출에 실려 나가는 비용이자 정확도**입니다.

## 실험 2 — `tool_choice`: 부를지 말지 누가 정하나

기본값은 모델이 정합니다(`auto`) — 이게 ReAct의 요지입니다(`classify_one.py`의 `extract_react`가 그렇습니다).
하지만 "이번엔 무조건 이 도구를 써라"가 필요할 때가 있습니다(구조화 추출, 라우팅 강제 등).

**함정.** `create_agent(...)`에는 `tool_choice` 인자가 **없습니다**. 강제하려면 한 겹 내려가
`model.bind_tools(tools, tool_choice=...)`로 모델에 직접 겁니다. 값은 `"auto"`(모델이 결정) ·
`"required"`/`"any"`(아무 도구든 반드시) · 특정 도구 이름(그 도구를 콕 집어)입니다.

In [ ]:
if not HAS_KEY:
    print("이 셀은 키가 필요합니다. .env에 OPENROUTER_API_KEY를 넣고 다시 실행하세요.")
else:
    from langchain_openai import ChatOpenAI

    @tool
    def check_sum(items_total: float, printed_total: float) -> str:
        """영수증 항목합과 인쇄 총액이 맞는지 검산한다."""
        return "ok" if abs(items_total - printed_total) < 1 else "mismatch"

    llm = ChatOpenAI(model=MODEL, base_url=BASE, api_key=KEY, temperature=0)
    ask = "고마워요, 오늘 날씨 좋네요!"   # 도구와 무관한 인사 — 강제하지 않으면 부를 이유가 없다
    for tc in ["auto", "required", "check_sum"]:
        bound = llm.bind_tools([check_sum], tool_choice=tc)
        calls = [c["name"] for c in (bound.invoke(ask).tool_calls or [])]
        print(f"tool_choice={tc!r:11} → 호출한 도구: {calls or '(없음)'}")

**관찰.** 같은 인사 입력인데:
- `auto` → 모델이 "부를 이유 없음"이라 판단해 **호출 안 함**.
- `required` → 관계없어도 **아무 도구나 반드시** 부름.
- `"check_sum"` → 그 도구를 **콕 집어** 부름.

`tool_choice`는 모델의 자율(ReAct)과 개발자의 강제 사이 제어점입니다. 순서·분기를 내가 통제해야 하면
본문처럼 `StateGraph`로 내려가고, "이 노드에선 반드시 이 도구"가 필요하면 여기 `bind_tools(tool_choice=...)`를 씁니다.

> **✏️ 직접 해보기.** `ask`를 `"이 영수증 합계 좀 검산해줘: 항목합 4900, 총액 4900"`으로 바꾸면 `auto`도
> 스스로 `check_sum`을 부릅니다. 도구가 *필요한* 입력에선 `auto`와 `required`가 같아짐을 확인하세요.

## 실험 3 — structured output: 파싱을 라이브러리에 맡긴다

`classify_one.py`의 기본 추출(`extract_singleshot`)은 이렇게 끝납니다:

```python
raw = msg.content              # 모델이 낸 '텍스트'
return RecordV1.model_validate_json(_strip_fences(raw))   # ```json 울타리를 손으로 걷고 검증
```

모델이 앞뒤에 설명 한 줄이라도 붙이면 이 파싱은 깨집니다. `with_structured_output(Schema)`는 그 파싱을
라이브러리에 위임합니다 — langchain-openai 1.x 기본값 `method="json_schema"`로 스키마를 `response_format`(제공자의 네이티브
structured output)에 실어 모델 출력을 스키마에 맞게 제약하고, **검증된 타입 객체를 곧장** 돌려줍니다.
function/tool calling과는 다른 경로입니다 — `method="function_calling"`을 주면 스키마를 도구로 바인딩합니다.
먼저 작은 스키마로 감을 잡습니다.

In [ ]:
if not HAS_KEY:
    print("이 셀은 키가 필요합니다.")
else:
    from langchain_openai import ChatOpenAI

    class Expense(BaseModel):
        """거래 한 줄의 분류."""
        merchant: str = Field(description="상호명")
        category: str = Field(description="식비/교통/생활 중 하나")
        amount: int = Field(description="금액(원)")

    llm = ChatOpenAI(model=MODEL, base_url=BASE, api_key=KEY, temperature=0)
    typed = llm.with_structured_output(Expense)          # ← 파싱을 위임
    out = typed.invoke("GS25에서 도시락 4900원 결제")
    print(type(out).__name__, "→", out)                  # str이 아니라 Expense 객체
    print("out.category =", out.category)                # 필드 접근이 바로 된다(파싱 없음)

이제 **진짜 파이프라인 스키마**로. `classify_one`에는 이미 `with_structured_output(RecordV1)`을 쓰는
`extract_structured()`가 있습니다. 수동 파싱 버전(`extract_singleshot`)과 나란히 돌려 결과가 같은지 봅니다.

In [ ]:
if not HAS_KEY:
    print("이 셀은 키가 필요합니다.")
else:
    from classify_one import extract_singleshot, extract_structured

    doc = "receipt_gs25.png"
    manual = extract_singleshot(doc, MODEL)     # 수동 JSON 파싱 경로
    typed = extract_structured(doc, MODEL)      # with_structured_output 경로
    print("수동 파싱   :", manual.merchant, f"{manual.total:,.0f}원", f"{len(manual.items)}항목")
    print("structured :", typed.merchant, f"{typed.total:,.0f}원", f"{len(typed.items)}항목")
    print("두 경로 동일?", manual.merchant == typed.merchant and abs(manual.total - typed.total) < 1)

**관찰.** 두 경로의 결과는 같습니다 — 하지만 코드가 다릅니다. structured output 경로엔
`_strip_fences`도 `model_validate_json`도 없습니다. 파싱 실패 처리를 라이브러리가 가져갑니다.
터미널에서도 확인할 수 있습니다:

```bash
uv run python3 ch1-llm-basics/classify_one.py --doc receipt_gs25.png --structured
```

> **참고.** `create_agent(response_format=RecordV1)`도 같은 원리입니다 — 에이전트 루프의 *마지막*에
> 구조화 응답을 강제할 때 씁니다. 내부적으로 `ToolStrategy`(도구로 스키마 전달) 또는
> `ProviderStrategy`(제공자의 네이티브 structured output = `response_format`)를 골라 씁니다.
>
> **✏️ 직접 해보기.** 위 `Expense`에 `date: str = Field(description="YYYY-MM-DD")` 필드를 더하고
> 프롬프트에 날짜를 넣어 다시 실행하세요. 스키마를 늘리면 타입 객체도 함께 자라는 걸 확인합니다.

## 정리 — 세 제어점이 파이프라인 어디에 있는가

| 제어점 | 실무 질문 | 우리 코드의 실물 |
|---|---|---|
| `@tool` args_schema | 인자 의미를 모델에 어떻게 정확히 전달? | `classify_one.py` `check_receipt_sum` = `@tool(args_schema=CheckReceiptSumInput)` |
| `tool_choice` | 도구 호출을 강제/금지하려면? | `bind_tools([...], tool_choice=...)` (ReAct는 `auto`) |
| structured output | 모델 출력을 타입으로 받으려면? | `classify_one.py` `extract_structured` = `with_structured_output(RecordV1)` |

`create_agent`는 이 셋을 감춰 편하지만, 강제·정밀 제어가 필요한 순간 한 겹 내려가야 합니다.
그 아래가 방금 만진 `bind_tools` · `with_structured_output` · `args_schema`입니다.